In [1]:
! pip install -r requirements.txt

  Using cached altair-6.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached argon2_cffi_bindings-25.1.0-cp39-abi3-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (7.4 kB)
  Using cached arrow-1.4.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached async_lru-2.3.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached babel-2.18.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached bleach-6.3.0-py3-none-any.whl.metadata (31 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-7.1.4-py3-none-any.whl.metadata (5.5 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached cffi-2.0.0-cp31

# 🤖 Train Multi-Horizon Model

We will train a multi-output regression model to predict stock close prices for multiple horizons (1, 3, 5, and 10 days ahead). The model will be trained fusioning both technical indicators from the price data and sentiment features extracted from the news articles. We will evaluate the model's performance using appropriate regression metrics to understand its predictive capabilities and limitations.

In [2]:
from src.data_loading import align_news_to_price
from src.data_insights import load_data
from src.feature_engineering import prepare_features, get_llm_embeddings, scale_features
from src.model import train_model, save_model
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, r2_score

2026-05-30 23:28:09.223 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:28:09.745 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
/workspaces/news-sentiment-analysis-for-stock-price-prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-30 23:28:25.659 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:28:25.661 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:28:25.662 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:28:25.663 WARNING streamlit.runtime.caching.cache_data_api: No 

## Prepare data

In [3]:
price_df, news_df = load_data()

df_aligned = align_news_to_price(price_df, news_df)

df_with_emb = get_llm_embeddings(df_aligned)

# Prepare features
df_ready, numeric_cols = prepare_features(df_with_emb)

joblib.dump((df_ready, numeric_cols), 'models/prepared_data.pkl')

# Split train/test
split_date = pd.to_datetime('2024-01-01')
train_df = df_ready[df_ready['date'] < split_date]
test_df = df_ready[df_ready['date'] >= split_date]

target_horizons = [1, 3, 5, 10]
y_train = train_df[[f'target_return_{h}d' for h in target_horizons]].values
y_test = test_df[[f'target_return_{h}d' for h in target_horizons]].values

2026-05-30 23:28:31.509 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:28:31.510 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 23:28:31.640 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:28:31.738 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 23:28:34.018 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:28:34.046 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
Loading weights: 100%|██████████| 103/103 [00:02<00:00,

Generating embeddings for 1428 unique news texts...


2026-05-30 23:30:51.586 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:30:51.620 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 23:30:51.621 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:30:51.698 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


## Scale Features

In [4]:
# Scale features
X_train_scaled, X_test_scaled, scaler = scale_features(
    train_df, test_df, numeric_cols
)

# Save scaler
joblib.dump(scaler, 'models/scaler.pkl')

2026-05-30 23:31:01.490 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 23:31:01.543 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


['models/scaler.pkl']

## Training

In [5]:
model = train_model(X_train_scaled, y_train)
save_model(model, 'models/regression_model.pkl')

Training on 161 samples with 403 features
Model saved to models/regression_model.pkl


## Evaluate

In [6]:
# Evaluate
y_pred_train = model.predict(X_train_scaled)
y_pred = model.predict(X_test_scaled)

## Results

In [7]:
results_df_train_set = pd.DataFrame({
    'Horizon (days)': target_horizons,
    'MAE': [mean_absolute_error(y_train[:, i], y_pred_train[:, i])
            for i in range(len(target_horizons))],
    'R2': [r2_score(y_train[:, i], y_pred_train[:, i])
           for i in range(len(target_horizons))]
})

results_df_train_set

,Horizon (days),MAE,R2
0,1,0.000357,0.998878
1,3,0.000321,0.999681
2,5,0.000358,0.999726
3,10,0.000362,0.999852


In [8]:
results_df_test_set = pd.DataFrame({
    'Horizon (days)': target_horizons,
    'MAE': [mean_absolute_error(y_test[:, i], y_pred[:, i])
            for i in range(len(target_horizons))],
    'R2': [r2_score(y_test[:, i], y_pred[:, i])
           for i in range(len(target_horizons))]
})

results_df_test_set

,Horizon (days),MAE,R2
0,1,0.016922,-0.047428
1,3,0.029893,-0.072421
2,5,0.039573,-0.077896
3,10,0.057335,-0.050335


In [9]:
feature_importance = model.estimators_[0].feature_importances_
top_features = sorted(zip(numeric_cols, feature_importance), 
                      key=lambda x: x[1], reverse=True)[:10]

top_df = pd.DataFrame(top_features, columns=['Feature', 'Importance']).set_index('Feature')

top_df

,Importance
Feature,
news_emb_239,0.031457
news_emb_229,0.016730
news_emb_316,0.015864
news_emb_307,0.014664
news_emb_376,0.012764
news_emb_363,0.012577
news_emb_169,0.011891
news_emb_369,0.011702
news_emb_121,0.010276
